Project 2 Jupyter Notebook
========

### Group: James Lind
Authors: Ivan Chan, Isha Chhabra, Matthew Arcaina, Kira Obsitnik, Zixu Yang

In this notebook, we'll solve the minimum cost diet problem using a set of procedures discussed in previous lectures. 

The food prices dataset we constructed is based on Berkeley's Whole Foods Store Data. We selected roughly 190 food items, with the majority of them being vegan. We also utilized the class's Dietary Requirements spreadsheet for our dietary requirements, as our targeted population is Vegan UC Berkeley Students. Then we will map our foods to their corresponding nutritional information from the USDA fooddatacentral API.

### Set up & Imports
Below are all necessary imports and pip installs for the rest of this project. 

**Please ensure all neccessary files are in the folder before running the below pip install.**

In [5]:
%pip install -r requirements.txt --upgrade

  Using cached pint-0.25.2-py3-none-any.whl.metadata (10 kB)
  Using cached numpy-2.4.2-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.1-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached gspread-6.2.1-py3-none-any.whl.metadata (11 kB)
  Using cached gspread_pandas-3.3.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached bottleneck-1.6.0-cp311-cp311-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (8.2 kB)
  Using cached eep153_tools-0.12.4-py2.py3-none-any.whl.metadata (363 bytes)
  Using cached fooddatacentral-1.0.12-py3-none-any.whl.metadata (2.1 kB)
  Using cached python_gnupg-0.5.6-py2.py3-none-any.whl.metadata (2.1 kB)
  Using cached flexcache-0.3-py3-none-any.whl.metadata (7.0 kB)
  Using

In [6]:
apikey = "NMhO744t7FjAKAlWrA1RCM6jVO07YE3kRTg1TlPo"  
# You could replace it with your key!

In [10]:
import fooddatacentral as fdc
import pandas as pd
from eep153_tools.sheets import read_sheets
import warnings
from  scipy.optimize import linprog as lp
import numpy as np

## Recap for the Problem
Below are explanation in matrix algebra of our problem, this is adopt from previous lectures.

We&rsquo;re thinking about the problem of finding the cheapest possible
nutritious diet.  Last time we argued that this problem could be
expressed as a *linear program*
$$
    \min_x p'x
$$

such that
$$\begin{bmatrix}
      A\\
      -A
   \end{bmatrix}x \geq \begin{bmatrix}
                        b_{min}\\
                        -b_{max}
                      \end{bmatrix},$$

where $p$ is a vector of prices, $A$ is a matrix that maps
vectors of quantities of food into vectors of nutrients, and where
$b_{min}$ and $b_{max}$ are respectively dietary minimums
and maximums of different nutrients.  As above, we will sometimes stack these
objects, obtaining
$$
      \tilde{A} = \begin{bmatrix}
                        A_{min}\\
                        -A_{max}
                      \end{bmatrix}
  $$
and
$$
      \tilde{b} = \begin{bmatrix}
                        b_{min}\\
                        -b_{max}
                      \end{bmatrix}
  $$

In this notebook: We will the objects required by the linear
program $(p,\tilde{A},\tilde{b})$, then have the computer solve the problem for us.



## Food Prices Spreadsheet, <span style="color: red;">*Deliverable [A] Data on prices for different foods* <a name="population-f"></span></a>

The code below allows us to collect data on different kinds of food with their prices from google spreadsheets. 

We have recorded the selected food items in a Google Spreadsheet: [https://docs.google.com/spreadsheets/d/18gdMQzetIll41oNgJtt3VgBpHgVn3zx57Agp89P28co/edit?usp=sharing](https://docs.google.com/spreadsheets/d/18gdMQzetIll41oNgJtt3VgBpHgVn3zx57Agp89P28co/)

In [40]:
df = read_sheets("18gdMQzetIll41oNgJtt3VgBpHgVn3zx57Agp89P28co",sheet='Whole Foods')

df = df.set_index('Food').drop(columns=['Brand'])

df

,Quantity,Units,Price,FDC,Vegan
Food,,,,,
Strawberries,1.0,lbs,4.99,2346409,NaN
Natures Reward Cilantro,3.0,oz,1.59,2709782,NaN
Blueberries,1.0,pint,5.29,2346411,NaN
Lime Regular Conventional,1.0,lbs,1.18,2709170,NaN
Red Raspberries,6.0,oz,3.99,2346410,NaN
...,...,...,...,...,...
Frozen Chicken Nuggets Plant Based,13.5,oz,10.99,2664241,NaN
Beyond Meat Stack Burger Plant-Based Patties,20.0,oz,16.99,2367272,NaN
Beyond Meat Beyond Sausage Plant-Based Dinner Sausage Links Brat Original,14.0,oz,7.87,2156900,NaN


### Converting Units

Below we will convert all food in either hundreds of grams (hectograms) or hundreds of milliliters (deciliters). This is for later application in FDC database in relation with nutrition information. 

**Code is adpoted from lecture notebook.**

In [43]:
# Convert food quantities to FDC units
df['FDC Quantity'] = df[['Quantity','Units']].T.apply(lambda x : fdc.units(x['Quantity'],x['Units']))

# Now divide price by the FDC Quantity to get, e.g., price per hectoliter
df['FDC Price'] = df['Price']/df['FDC Quantity']

df.dropna(how='any') # Drop food with any missing data

# To use minimum price observed
Prices = df.groupby('Food')['FDC Price'].min()

Prices

Food
365 by Whole Foods Market Chickenless Nuggets               2.062992320081521 / hectogram
365 by Whole Foods Market Grated Parmesan                  2.8148621635765165 / hectogram
365 by Whole Foods Market Non Dairy Gouda Cheese Slices    1.9797511144202005 / hectogram
365 by Whole Foods Market Organic Peeled Garlic            2.9336178354734375 / hectogram
365 by Whole Foods Market Organic Trimmed Green Beans      1.1728592348235487 / hectogram
                                                                        ...              
Wonderful Pistachios In-Shell Unsalted Nuts                2.2024179992269266 / hectogram
Wonderful Pistachios No Shells, Lightly Salted Nuts          4.40630574686842 / hectogram
Wonderful Pistachios No Shells, Roasted & Salted Nuts        4.40630574686842 / hectogram
Yellow Nectarine                                           0.8796444261176615 / hectogram
Yogurt Coconut Chocolate Mousse Organic                    4.4338637267791645 / deciliter
Name:

## Foods Nutrition Information, <span style="color: red;">*Deliverable [A] Nutritional Content on different foods* <a name="population-f"></span></a>

In this delieverable, we will build the matrix $A$, which maps quantities of food into nutrients. Here we will do lookups on USDA database to get their nutritional information. 

**Code is adopted from lecture notebooks.**

In [42]:
D = {}
count = 0
for food in df.index:
    try:
        FDC = df.loc[df.index==food,:].FDC.values[0]
        count+=1
        D[food] = fdc.nutrients(apikey,FDC).Quantity
    except AttributeError:
        warnings.warn(f"Couldn't find FDC Code {FDC} for food {food}.")

D = pd.DataFrame(D,dtype=float)

D

,Strawberries,Natures Reward Cilantro,Blueberries,Lime Regular Conventional,Red Raspberries,PRODUCE Organic Mandarin Bag,Blackberries,Organic Rancher Organic Ground Beef 93% Lean/7% Fat,Organic Baby Spinach,Banana Conventional,...,Tofurky Original Deli Slices,Dr. Praeger's Perfect Burgers,Peppers Red Fire Roasted Organic,Frozen Burger Veggie Mushroom Mozzarella,Field Roast Signature Stadium Plant-Based Hot Dogs,Frozen Chicken Nuggets Plant Based,Beyond Meat Stack Burger Plant-Based Patties,Beyond Meat Beyond Sausage Plant-Based Dinner Sausage Links Brat Original,Louisville Vegan Jerky Sweet & Smoky Carolina Mustard BBQ Vegan Meat,"Dr. Praeger's, Mushroom Risotto Veggie Burgers"
5-methyl tetrahydrofolate (5-MTHF),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Alanine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.307,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"Alcohol, ethyl",NaN,0.0,NaN,0.00,NaN,NaN,NaN,0.000,NaN,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Amino acids,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Arginine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.358,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Vitamins and Other Components,0.000,NaN,0.00000,NaN,0.0000,0.00000,0.0000,0.000,0.0000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Water,90.830,92.2,84.19000,88.30,85.5500,84.65000,86.4500,71.720,92.5200,75.60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Zeaxanthin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,191.3000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"Zinc, Zn",0.114,0.5,0.08534,0.11,0.2213,0.06869,0.1889,4.970,0.4471,0.16,...,NaN,NaN,NaN,NaN,NaN,3.95,4.07,NaN,NaN,NaN
